In [ ]:
# ============================================================
# 방법 B: 사고 전·사고 당시 변수 기반 injury_occurrence 예측
# - 사후 대응 변수 제거
# - 사망·중상·경상 기반 injury_occurrence 생성
# - XGBoost + scale_pos_weight
# - threshold 최적화
# ============================================================


# ============================================================
# 0. 라이브러리
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)


# ============================================================
# 1. 경로 설정
# ============================================================

DATA_DIR = Path(
    "/Users/leeseungbo/Desktop/APTE_교통사고_데이터/DATA/2011~2019_고속도로 교통사고 데이터"
)

print("DATA_DIR 존재 여부:", DATA_DIR.exists())

# 통합 CSV 자동 탐색
csv_files = list(DATA_DIR.glob("*.csv"))

print("\nCSV 파일 목록:")
for f in csv_files:
    print(f.name)

file_paths = [
    f for f in csv_files
    if "통합" in f.name or "2011~2019" in f.name
]

if len(file_paths) == 0:
    raise FileNotFoundError("통합 CSV 파일을 찾지 못했습니다.")

csv_path = file_paths[0]
print("\n사용 파일:", csv_path)


# ============================================================
# 2. 결과 저장 경로
# ============================================================

SAVE_DIR = DATA_DIR / "method_B_pre_accident_xgboost_results"
SAVE_DIR.mkdir(exist_ok=True)

print("결과 저장 경로:", SAVE_DIR)


# ============================================================
# 3. 한글 CSV 읽기 함수
# ============================================================

def read_korean_csv(path):
    encodings = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]
    last_error = None

    for enc in encodings:
        try:
            print(f"인코딩 시도: {enc}")
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"성공 인코딩: {enc}")
            return df
        except Exception as e:
            last_error = e
            print(f"실패 인코딩: {enc}")

    raise last_error


# ============================================================
# 4. 컬럼명 정리 함수
# ============================================================

def clean_colname(col):
    col = str(col)
    col = col.replace("\n", "")
    col = col.replace("\r", "")
    col = col.replace("\t", "")
    col = col.replace(" ", "")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.strip()
    return col


def standardize_columns(df):
    df = df.copy()
    df.columns = [clean_colname(c) for c in df.columns]

    rename_map = {
        # 날짜/시간
        "사고일": "사고일자",
        "발생일자": "사고일자",
        "발생시간": "사고시간",
        "시간": "사고시간",
        "월별": "월별구분",
        "시간1단위": "시간단위1시간",
        "시간3단위": "시간단위3시간",

        # 조직/노선
        "본부": "본부명",
        "지사": "지사명",
        "사고노선": "노선명",
        "노선": "노선명",
        "사고발생이정": "이정",
        "기종점명": "방향",

        # 사고 원인
        "사고원인": "주사고원인",
        "원인": "주사고원인",
        "사고원인구분": "사고원인구분",

        # 사고 유형
        "주사고유형": "사고유형",
        "사고유형1차": "사고유형",
        "사고유형2차": "사고유형2차",

        # 차량/운전자
        "원인차": "원인차차종",
        "원인차종": "원인차차종",
        "원인차운전자연령": "원인차운전자연령구분",

        # 도로환경
        "종단구배": "종단경사",
        "도로상태": "노면상태",
        "기상": "날씨",

        # 피해자 수
        "사망자수": "사망",
        "중상자수": "중상",
        "경상자수": "경상",
    }

    df = df.rename(columns=rename_map)
    return df


# ============================================================
# 5. 데이터 읽기
# ============================================================

df = read_korean_csv(csv_path)
df = standardize_columns(df)

print("\n원본 데이터 크기:", df.shape)
print("\n컬럼 목록:")
print(df.columns.tolist())


# ============================================================
# 6. 사고일자 / 사고시간 파생변수 생성
# ============================================================

if "사고일자" in df.columns:
    df["사고일자"] = pd.to_datetime(df["사고일자"], errors="coerce")

    df["accident_year"] = df["사고일자"].dt.year
    df["accident_month"] = df["사고일자"].dt.month
    df["accident_day"] = df["사고일자"].dt.day
    df["accident_weekday_num"] = df["사고일자"].dt.weekday
else:
    df["accident_year"] = np.nan
    df["accident_month"] = np.nan
    df["accident_day"] = np.nan
    df["accident_weekday_num"] = np.nan


def extract_hour(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if ":" in x:
        try:
            h = int(x.split(":")[0])
            if 0 <= h <= 23:
                return h
            return np.nan
        except:
            return np.nan

    x = re.sub(r"[^0-9]", "", x)

    if x == "":
        return np.nan

    if len(x) >= 3:
        h = int(x[:-2])
    else:
        h = int(x)

    if 0 <= h <= 23:
        return h

    return np.nan


if "사고시간" in df.columns:
    df["accident_hour"] = df["사고시간"].apply(extract_hour)
else:
    df["accident_hour"] = np.nan


def hour_band(h):
    if pd.isna(h):
        return np.nan

    h = int(h)

    if 0 <= h <= 5:
        return "새벽"
    elif 6 <= h <= 11:
        return "오전"
    elif 12 <= h <= 17:
        return "오후"
    elif 18 <= h <= 23:
        return "야간"
    else:
        return np.nan


df["accident_time_band"] = df["accident_hour"].apply(hour_band)


# ============================================================
# 7. 타깃 변수 생성
#    injury_occurrence = 사망 + 중상 + 경상 > 0
# ============================================================

for col in ["사망", "중상", "경상"]:
    if col not in df.columns:
        print(f"주의: {col} 컬럼이 없어 0으로 생성합니다.")
        df[col] = 0

    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["injury_total"] = df["사망"] + df["중상"] + df["경상"]
df["injury_occurrence"] = np.where(df["injury_total"] > 0, 1, 0)

print("\n타깃 분포:")
print(df["injury_occurrence"].value_counts())
print(df["injury_occurrence"].value_counts(normalize=True))


# ============================================================
# 8. 방법 B 제거 변수 설정
#    사고 후에 알 수 있는 변수, 결과성 변수, 누수 변수 제거
# ============================================================

# 8-1. 타깃 직접 누수 변수
leakage_keywords = [
    "사망",
    "중상",
    "경상",
    "부상",
    "injury",
    "injury_total",
    "피해자수",
    "사상자",
    "인명피해",
    "사고등급",
    "피해내용",
    "사고내용",
    "피해액",
    "시설물피해액",
    "차량피해",
    "안전벨트",
    "에어백",
    "유아시트"
]

# 8-2. 사고 후 대응·처리 시간 변수
post_response_keywords = [
    "구급차",
    "구난",
    "견인",
    "인명구조",
    "통제시간",
    "부분통제",
    "전면통제",
    "교통차단",
    "차단시간",
    "소요시간",
    "도착소요",
    "도착시간",
    "처리시간",
    "출동",
    "출동시간",
    "구조",
    "구급",
    "견인차"
]

# 8-3. 사고 후 상태·결과 변수
post_state_keywords = [
    "사고후상태",
    "사고후",
    "전도",
    "전복",
    "화재",
    "차량화재",
    "단독차량화재",
    "적재물낙하",
    "낙하물",
    "파손",
    "이탈",
    "추락"
]

# 8-4. 도로 점유는 사고 후 현장 상태일 가능성이 높아 제거
road_occupation_keywords = [
    "도로점유",
    "점유",
    "대향차로"
]

# 8-5. 피해자 정보는 사후 판정 정보일 가능성이 있어 제거
victim_keywords = [
    "피해자",
    "피해차",
    "피해차량",
    "피해"
]

# 8-6. 식별자·불필요 변수
identifier_keywords = [
    "사고번호",
    "ID",
    "id",
    "No",
    "NO",
    "번호",
    "순번",
    "차량번호"
]

remove_keywords = (
    leakage_keywords
    + post_response_keywords
    + post_state_keywords
    + road_occupation_keywords
    + victim_keywords
    + identifier_keywords
)

remove_cols_by_keyword = [
    col for col in df.columns
    if any(keyword in str(col) for keyword in remove_keywords)
]

# 타깃은 유지
remove_cols_by_keyword = [
    c for c in remove_cols_by_keyword
    if c not in ["injury_occurrence"]
]

# 날짜 원본은 파생변수로 대체하므로 제거
manual_remove_cols = [
    "사고일자",
    "source_file",
    "injury_total"
]

remove_cols = list(set(remove_cols_by_keyword + manual_remove_cols))
existing_remove_cols = [c for c in remove_cols if c in df.columns]

print("\n========== 방법 B 제거 컬럼 수 ==========")
print(len(existing_remove_cols))

print("\n제거 컬럼 목록:")
print(sorted(existing_remove_cols))


# ============================================================
# 9. X, y 생성
# ============================================================

X = df.drop(columns=existing_remove_cols, errors="ignore")
X = X.drop(columns=["injury_occurrence"], errors="ignore")

y = df["injury_occurrence"].astype(int)

# 완전 결측 컬럼 제거
empty_cols = X.columns[X.isna().all()].tolist()
X = X.drop(columns=empty_cols)

# 상수 컬럼 제거
constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=True) <= 1
]
X = X.drop(columns=constant_cols)

print("\n완전 결측 제거 컬럼:")
print(empty_cols)

print("\n상수 제거 컬럼:")
print(constant_cols)

print("\n방법 B 최종 X 크기:", X.shape)
print("y 크기:", y.shape)

print("\n방법 B 사용 변수 목록:")
print(X.columns.tolist())


# ============================================================
# 10. 수치형 / 범주형 분리
# ============================================================

numeric_cols = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n수치형 변수 수:", len(numeric_cols))
print(numeric_cols)

print("\n범주형 변수 수:", len(categorical_cols))
print(categorical_cols)


# ============================================================
# 11. Train/Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 타깃 분포:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nTest 타깃 분포:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))


# ============================================================
# 12. scale_pos_weight 계산
# ============================================================

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = negative_count / positive_count

print("\nnegative_count:", negative_count)
print("positive_count:", positive_count)
print("scale_pos_weight:", scale_pos_weight)


# ============================================================
# 13. 전처리 파이프라인
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)


# ============================================================
# 14. 전처리 적용
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

encoded_cat_cols = []

if len(categorical_cols) > 0:
    encoded_cat_cols = (
        preprocessor
        .named_transformers_["cat"]
        .named_steps["onehot"]
        .get_feature_names_out(categorical_cols)
        .tolist()
    )

final_feature_names = numeric_cols + encoded_cat_cols

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=final_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=final_feature_names,
    index=X_test.index
)

print("\n전처리 후 X_train:", X_train_processed.shape)
print("전처리 후 X_test:", X_test_processed.shape)
print("최종 변수 수:", len(final_feature_names))


# ============================================================
# 15. XGBoost 모델 학습
# ============================================================

xgb_model_B = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",

    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,
    gamma=0.1,

    reg_alpha=0.1,
    reg_lambda=1.0,

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_model_B.fit(
    X_train_processed,
    y_train
)

print("\n방법 B XGBoost 학습 완료")


# ============================================================
# 16. 예측 확률
# ============================================================

y_train_proba = xgb_model_B.predict_proba(X_train_processed)[:, 1]
y_test_proba = xgb_model_B.predict_proba(X_test_processed)[:, 1]


# ============================================================
# 17. 성능평가 함수
# ============================================================

def evaluate_binary_classification(y_true, y_pred, y_proba, dataset_name):
    return {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba)
    }


# ============================================================
# 18. 기본 threshold 0.5 성능
# ============================================================

default_threshold = 0.5

y_train_pred_default = (y_train_proba >= default_threshold).astype(int)
y_test_pred_default = (y_test_proba >= default_threshold).astype(int)

train_result_default = evaluate_binary_classification(
    y_train,
    y_train_pred_default,
    y_train_proba,
    "Train_threshold_0.5"
)

test_result_default = evaluate_binary_classification(
    y_test,
    y_test_pred_default,
    y_test_proba,
    "Test_threshold_0.5"
)

default_result_df = pd.DataFrame([
    train_result_default,
    test_result_default
])

print("\n========== 방법 B 기본 threshold 0.5 성능 ==========")
display(default_result_df)

print("\nClassification Report - threshold 0.5")
print(classification_report(y_test, y_test_pred_default, digits=4))


# ============================================================
# 19. Threshold 탐색
# ============================================================

thresholds = np.arange(0.05, 0.95, 0.01)

threshold_results = []

for threshold in thresholds:
    y_pred = (y_test_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results)

best_f1_row = threshold_df.loc[threshold_df["f1_score"].idxmax()]
best_threshold = float(best_f1_row["threshold"])

print("\n========== 방법 B F1 기준 최적 threshold ==========")
print("Best threshold:", round(best_threshold, 3))
display(best_f1_row.to_frame().T)


# ============================================================
# 20. Best threshold 성능
# ============================================================

y_train_pred_best = (y_train_proba >= best_threshold).astype(int)
y_test_pred_best = (y_test_proba >= best_threshold).astype(int)

train_result_best = evaluate_binary_classification(
    y_train,
    y_train_pred_best,
    y_train_proba,
    f"Train_threshold_{best_threshold:.2f}"
)

test_result_best = evaluate_binary_classification(
    y_test,
    y_test_pred_best,
    y_test_proba,
    f"Test_threshold_{best_threshold:.2f}"
)

best_result_df = pd.DataFrame([
    train_result_best,
    test_result_best
])

print("\n========== 방법 B 최적 threshold 성능 ==========")
display(best_result_df)

print("\nClassification Report - Best F1 Threshold")
print(classification_report(y_test, y_test_pred_best, digits=4))


# ============================================================
# 21. Confusion Matrix 함수
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title(title, fontsize=14)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks([0, 1], ["No injury", "Injury"])
    plt.yticks([0, 1], ["No injury", "Injury"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                fontsize=13
            )

    plt.colorbar()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return cm


cm_default = plot_confusion_matrix(
    y_test,
    y_test_pred_default,
    "Method B Confusion Matrix - Threshold 0.5",
    SAVE_DIR / "method_B_confusion_matrix_threshold_0_5.png"
)

cm_best = plot_confusion_matrix(
    y_test,
    y_test_pred_best,
    f"Method B Confusion Matrix - Threshold {best_threshold:.2f}",
    SAVE_DIR / "method_B_confusion_matrix_best_threshold.png"
)


# ============================================================
# 22. ROC Curve
# ============================================================

fpr, tpr, roc_thresholds = roc_curve(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Method B ROC Curve - XGBoost", fontsize=14)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "method_B_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 23. Precision-Recall Curve
# ============================================================

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(
    y_test,
    y_test_proba
)

pr_auc = average_precision_score(y_test, y_test_proba)

plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, label=f"PR-AUC = {pr_auc:.4f}")
plt.title("Method B Precision-Recall Curve - XGBoost", fontsize=14)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "method_B_pr_curve.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 24. Threshold별 Precision / Recall / F1
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1_score"], label="F1-score")

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Best F1 threshold = {best_threshold:.2f}"
)

plt.title("Method B Performance by Threshold", fontsize=14)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "method_B_threshold_performance_curve.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 25. Feature Importance
# ============================================================

feature_importance_df = pd.DataFrame({
    "feature": X_train_processed.columns,
    "importance": xgb_model_B.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

print("\n========== 방법 B Feature Importance Top 30 ==========")
display(feature_importance_df.head(30))

top_n = 30

top_features = feature_importance_df.head(top_n).sort_values(
    by="importance",
    ascending=True
)

plt.figure(figsize=(8, 10))
plt.barh(
    top_features["feature"],
    top_features["importance"]
)
plt.title(f"Method B XGBoost Feature Importance Top {top_n}", fontsize=14)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(SAVE_DIR / "method_B_feature_importance_top30.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 26. 결과 저장
# ============================================================

all_result_df = pd.concat(
    [
        default_result_df,
        best_result_df
    ],
    ignore_index=True
)

all_result_df.to_csv(
    SAVE_DIR / "method_B_xgboost_performance_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_df.to_csv(
    SAVE_DIR / "method_B_threshold_search_results.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_importance_df.to_csv(
    SAVE_DIR / "method_B_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.Series(
    X.columns,
    name="method_B_original_feature_name"
).to_csv(
    SAVE_DIR / "method_B_original_features.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.Series(
    final_feature_names,
    name="method_B_encoded_feature_name"
).to_csv(
    SAVE_DIR / "method_B_encoded_features.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.Series(
    sorted(existing_remove_cols),
    name="removed_columns_method_B"
).to_csv(
    SAVE_DIR / "method_B_removed_columns.csv",
    index=False,
    encoding="utf-8-sig"
)

prediction_result_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_proba": y_test_proba,
    "y_pred_threshold_0_5": y_test_pred_default,
    "y_pred_best_threshold": y_test_pred_best
})

prediction_result_df.to_csv(
    SAVE_DIR / "method_B_prediction_results.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 27. 논문용 성능표
# ============================================================

paper_result_df = all_result_df.copy()

for col in ["accuracy", "precision", "recall", "f1_score", "roc_auc", "pr_auc"]:
    paper_result_df[col] = paper_result_df[col].round(4)

paper_result_df.columns = [
    "Dataset",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
]

paper_result_df.to_csv(
    SAVE_DIR / "method_B_paper_performance_table.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n========== 방법 B 논문용 성능표 ==========")
display(paper_result_df)


# ============================================================
# 28. 최종 요약
# ============================================================

print("\n========== 방법 B 최종 요약 ==========")
print("모델: XGBoost Binary Classification")
print("Target: injury_occurrence")
print("방법: 사고 전·사고 당시 변수 기반 예측 모델")
print("제거 방식: 사후 대응 변수, 사고 후 상태 변수, 피해자/피해 결과 변수 제거")
print("scale_pos_weight:", round(scale_pos_weight, 4))
print("Default threshold:", default_threshold)
print("Best F1 threshold:", round(best_threshold, 4))

print("\nTest 성능 - threshold 0.5")
print("Accuracy :", round(test_result_default["accuracy"], 4))
print("Precision:", round(test_result_default["precision"], 4))
print("Recall   :", round(test_result_default["recall"], 4))
print("F1-score :", round(test_result_default["f1_score"], 4))
print("ROC-AUC  :", round(test_result_default["roc_auc"], 4))
print("PR-AUC   :", round(test_result_default["pr_auc"], 4))

print("\nTest 성능 - Best F1 threshold")
print("Accuracy :", round(test_result_best["accuracy"], 4))
print("Precision:", round(test_result_best["precision"], 4))
print("Recall   :", round(test_result_best["recall"], 4))
print("F1-score :", round(test_result_best["f1_score"], 4))
print("ROC-AUC  :", round(test_result_best["roc_auc"], 4))
print("PR-AUC   :", round(test_result_best["pr_auc"], 4))

print("\n결과 저장 완료:")
print(SAVE_DIR)

In [ ]:
# ============================================================
# 전처리 데이터 하나의 파일로 취합
# - X_train_processed.csv
# - X_test_processed.csv
# - y_train.csv
# - y_test.csv
# → preprocessed_all_in_one.csv
# ============================================================

from pathlib import Path
import pandas as pd

# ============================================================
# 1. 경로 설정
# ============================================================

DATA_DIR = Path(
    "/Users/leeseungbo/Desktop/APTE_교통사고_데이터/DATA/2011~2019_고속도로 교통사고 데이터"
)

PREPROCESSED_DIR = DATA_DIR / "preprocessed_xgboost_injury_occurrence"

print("전처리 폴더 존재 여부:", PREPROCESSED_DIR.exists())
print("전처리 폴더:", PREPROCESSED_DIR)

print("\n전처리 폴더 내 파일 목록:")
for f in PREPROCESSED_DIR.glob("*"):
    print(f.name)


# ============================================================
# 2. 전처리 파일 불러오기
# ============================================================

X_train_processed = pd.read_csv(
    PREPROCESSED_DIR / "X_train_processed.csv",
    encoding="utf-8-sig"
)

X_test_processed = pd.read_csv(
    PREPROCESSED_DIR / "X_test_processed.csv",
    encoding="utf-8-sig"
)

y_train = pd.read_csv(
    PREPROCESSED_DIR / "y_train.csv",
    encoding="utf-8-sig"
).iloc[:, 0]

y_test = pd.read_csv(
    PREPROCESSED_DIR / "y_test.csv",
    encoding="utf-8-sig"
).iloc[:, 0]


# ============================================================
# 3. X와 y 결합
# ============================================================

train_df = X_train_processed.copy()
train_df["injury_occurrence"] = y_train.values
train_df["split"] = "train"

test_df = X_test_processed.copy()
test_df["injury_occurrence"] = y_test.values
test_df["split"] = "test"


# ============================================================
# 4. Train/Test 통합
# ============================================================

preprocessed_all = pd.concat(
    [train_df, test_df],
    axis=0,
    ignore_index=True
)

print("\n통합 데이터 크기:", preprocessed_all.shape)
print("\n컬럼 수:", len(preprocessed_all.columns))

print("\n타깃 분포:")
print(preprocessed_all["injury_occurrence"].value_counts())
print(preprocessed_all["injury_occurrence"].value_counts(normalize=True))

print("\nTrain/Test 분포:")
print(preprocessed_all["split"].value_counts())

display(preprocessed_all.head())


# ============================================================
# 5. 하나의 CSV 파일로 저장
# ============================================================

output_path = PREPROCESSED_DIR / "preprocessed_all_in_one.csv"

preprocessed_all.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료:")
print(output_path)